In [0]:
import mlflow

# This automatically gets your username
username = spark.sql("SELECT current_user()").first()[0]
experiment_path = f"/Users/{username}/safetrack-risk-model"

mlflow.set_experiment(experiment_path)

with mlflow.start_run():
    mlflow.log_param("test_param", 42)
    mlflow.log_metric("test_metric", 0.99)
    print("MLflow is working!")
    print(f"Experiment path: {experiment_path}")

In [0]:
from pyspark.sql import functions as F

df = spark.read.csv(
    "/Volumes/workspace/default/safetrack_data/etrain_delays.csv",
    header=True,
    inferSchema=True
)

df.write.format("delta").mode("overwrite") \
  .option("delta.columnMapping.mode", "name") \
  .saveAsTable("workspace.default.running_history")

print("✅ running_history created!")
print(f"Total rows: {df.count()}")
df.printSchema()
display(df)

In [0]:
from pyspark.sql import functions as F

# Read the CSV
df = spark.read.csv(
    "/Volumes/workspace/default/safetrack_data/RS_Session_260_AU_1894_A.csv",
    header=True,
    inferSchema=True
)

# Clean column names - remove special characters
clean_cols = []
for col in df.columns:
    new_col = col.strip()
    new_col = new_col.replace(" ", "_")
    new_col = new_col.replace("(", "")
    new_col = new_col.replace(")", "")
    new_col = new_col.replace(".", "")
    new_col = new_col.replace(",", "")
    new_col = new_col.replace("-", "_")
    clean_cols.append(new_col)

# Apply cleaned column names
df = df.toDF(*clean_cols)

print("Cleaned columns:", df.columns)

# Save as Delta table
df.write.format("delta").mode("overwrite") \
  .option("delta.columnMapping.mode", "name") \
  .saveAsTable("workspace.default.incidents")

print("✅ Table created successfully!")
print(f"Total rows: {df.count()}")
display(df)

In [0]:
# Check what columns the incidents table actually has
df = spark.table("workspace.default.incidents")
df.printSchema()
display(df)

In [0]:
from pyspark.sql import functions as F

# Weather correlation per zone
df_weather = spark.table("weather_history") \
    .filter(F.col("zone").isNotNull())

df_weather_feat = df_weather.groupBy("zone").agg(
    F.avg("precipitation_mm").alias("avg_precip")
).withColumnRenamed("zone", "Zonal_Railway")

max_precip = df_weather_feat.agg(F.max("avg_precip")).first()[0] or 1
df_weather_feat = df_weather_feat.withColumn(
    "weather_correlation",
    F.round(F.col("avg_precip") / max_precip, 4)
).select("Zonal_Railway", "weather_correlation")

# Incident proximity from real data
df_inc = spark.table("workspace.default.incidents_by_zone") \
    .filter(F.col("Zonal_Railway") != "Total") \
    .withColumn(
        "total_incidents",
        F.col("Number_of_Consequential_Train_Collisions") +
        F.col("Number_of_Consequential_Train_Accidents_on_Account_of_SPAD")
    )

max_inc = df_inc.agg(F.max("total_incidents")).first()[0] or 1
df_inc_feat = df_inc.withColumn(
    "incident_proximity",
    F.round(F.col("total_incidents") / max_inc, 4)
).select("Zonal_Railway", "incident_proximity")

# Delay spike freq from existing table
df_delay_feat = spark.table("workspace.default.features_per_segment") \
    .select("Zonal_Railway", "delay_spike_freq")

# All zones scaffold
all_zones = [
    ("Central Railway",), ("Eastern Railway",), ("East Central Railway",),
    ("East Coast Railway",), ("Konkan Railway",), ("North Central Railway",),
    ("North Eastern Railway",), ("Northeast Frontier Railway",),
    ("North Western Railway",), ("Northern Railway",), ("South Central Railway",),
    ("South Eastern Railway",), ("South East Central Railway",),
    ("South Western Railway",), ("Southern Railway",), ("West Central Railway",),
    ("Western Railway",), ("Metro Railway Kolkata",), ("Production Units",),
]
df_all = spark.createDataFrame(all_zones, ["Zonal_Railway"])

# Join all three features
df_features = df_all \
    .join(df_delay_feat,   on="Zonal_Railway", how="left") \
    .join(df_weather_feat, on="Zonal_Railway", how="left") \
    .join(df_inc_feat,     on="Zonal_Railway", how="left")

# Fill gaps with means
mean_delay   = df_delay_feat.agg(F.avg("delay_spike_freq")).first()[0] or 0.4
mean_weather = df_weather_feat.agg(F.avg("weather_correlation")).first()[0] or 0.5
mean_inc     = df_inc_feat.agg(F.avg("incident_proximity")).first()[0] or 0.3

df_features = df_features.fillna({
    "delay_spike_freq":    round(mean_delay, 4),
    "weather_correlation": round(mean_weather, 4),
    "incident_proximity":  round(mean_inc, 4),
})

print(f"✅ Real features table — {df_features.count()} zones")
df_features.orderBy("Zonal_Railway").show(25, truncate=False)

# Overwrite with all 3 real features
df_features.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.features_per_segment")

print("✅ features_per_segment now has all 3 real features!")

In [0]:
import mlflow
from pyspark.sql import functions as F

# Score every zone
df_scored = spark.table("workspace.default.features_per_segment") \
    .withColumn(
        "risk_score",
        F.round(
            F.col("delay_spike_freq")    * 0.4 +
            F.col("weather_correlation") * 0.3 +
            F.col("incident_proximity")  * 0.3,
        4)
    ).withColumn(
        "risk_label",
        F.when(F.col("risk_score") >= 0.6, "High")
         .when(F.col("risk_score") >= 0.4, "Medium")
         .otherwise("Low")
    )

df_scored.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.risk_scores")

print("All zones scored:")
df_scored.orderBy("risk_score", ascending=False).show(25, truncate=False)

# Back-test: do High-risk zones have real incidents?
df_inc_check = spark.table("workspace.default.incidents_by_zone") \
    .filter(F.col("Zonal_Railway") != "Total") \
    .withColumn(
        "total_incidents",
        F.col("Number_of_Consequential_Train_Collisions") +
        F.col("Number_of_Consequential_Train_Accidents_on_Account_of_SPAD")
    ).select("Zonal_Railway", "total_incidents")

df_high = df_scored.filter(F.col("risk_label") == "High")
df_backtest = df_high.join(df_inc_check, on="Zonal_Railway", how="left") \
    .fillna({"total_incidents": 0})

total_high = df_backtest.count()
confirmed  = df_backtest.filter(F.col("total_incidents") > 0).count()
accuracy   = round((confirmed / total_high * 100), 1) if total_high > 0 else 0

print("=" * 55)
print("   SAFETRACK RISK MODEL — BACK-TEST RESULTS")
print("=" * 55)
print(f"High-risk zones identified  : {total_high}")
print(f"Confirmed by real incidents : {confirmed}")
print(f"✅ BACK-TEST ACCURACY       : {accuracy}%")
print()
print("High-risk zone breakdown:")
df_backtest.orderBy("risk_score", ascending=False) \
    .select("Zonal_Railway", "risk_score", "risk_label", "total_incidents") \
    .show(truncate=False)

# Log real metrics to MLflow
username = spark.sql("SELECT current_user()").first()[0]
mlflow.set_experiment(f"/Users/{username}/safetrack-risk-model")

with mlflow.start_run(run_name="safetrack-real-backtest"):
    mlflow.log_param("model_type",          "weighted_scoring")
    mlflow.log_param("delay_weight",        0.4)
    mlflow.log_param("weather_weight",      0.3)
    mlflow.log_param("incident_weight",     0.3)
    mlflow.log_param("high_risk_threshold", 0.6)
    mlflow.log_metric("backtest_accuracy",  accuracy)
    mlflow.log_metric("high_risk_zones",    total_high)
    mlflow.log_metric("confirmed_zones",    confirmed)
    print(f"✅ Real metrics logged to MLflow — accuracy: {accuracy}%")

In [0]:
# Check 1 — confirm weather_history column names
spark.table("weather_history").printSchema()

# Check 2 — confirm incidents_by_zone column names  
spark.table("workspace.default.incidents_by_zone").printSchema()

# Check 3 — confirm running_history column names
spark.table("workspace.default.running_history").printSchema()

In [0]:
spark.table("workspace.default.features_per_segment").printSchema()

In [0]:
# Load zone-wise accidents data
df_zone = spark.read.csv(
    "/Volumes/workspace/default/safetrack_data/<RS_Session_260_AU_1886_A>.csv",
    header=True,
    inferSchema=True
)

# Clean column names
clean_cols = []
for col in df_zone.columns:
    new_col = col.strip().replace(" ", "_").replace("(", "").replace(")", "").replace(".", "").replace(",", "").replace("-", "_")
    clean_cols.append(new_col)

df_zone = df_zone.toDF(*clean_cols)

print("Columns:", df_zone.columns)
print("Rows:", df_zone.count())
display(df_zone)

In [0]:
# Load zone-wise accidents data
df_zone = spark.read.csv(
    "/Volumes/workspace/default/safetrack_data/RS_Session_260_AU_1886_A.csv",
    header=True,
    inferSchema=True
)

# Clean column names
clean_cols = []
for col in df_zone.columns:
    new_col = col.strip().replace(" ", "_").replace("(", "").replace(")", "").replace(".", "").replace(",", "").replace("-", "_")
    clean_cols.append(new_col)

df_zone = df_zone.toDF(*clean_cols)

print("Columns:", df_zone.columns)
print("Rows:", df_zone.count())
display(df_zone)

In [0]:
# Save as Delta table
df_zone.write.format("delta").mode("overwrite") \
  .option("delta.columnMapping.mode", "name") \
  .saveAsTable("workspace.default.incidents_by_zone")

print("✅ incidents_by_zone table created!")
print(f"Total zones: {df_zone.count()}")

In [0]:
# Verify the table
verify = spark.table("workspace.default.incidents_by_zone")
display(verify)

In [0]:
from pyspark.sql import functions as F

# Weather correlation per zone
df_weather = spark.table("weather_history") \
    .filter(F.col("zone").isNotNull())

df_weather_feat = df_weather.groupBy("zone").agg(
    F.avg("precipitation_sum").alias("avg_precip")
).withColumnRenamed("zone", "Zonal_Railway")

max_precip = df_weather_feat.agg(F.max("avg_precip")).first()[0] or 1
df_weather_feat = df_weather_feat.withColumn(
    "weather_correlation",
    F.round(F.col("avg_precip") / max_precip, 4)
).select("Zonal_Railway", "weather_correlation")

# Incident proximity from real data
df_inc = spark.table("workspace.default.incidents_by_zone") \
    .filter(F.col("Zonal_Railway") != "Total") \
    .withColumn(
        "total_incidents",
        F.col("Number_of_Consequential_Train_Collisions") +
        F.col("Number_of_Consequential_Train_Accidents_on_Account_of_SPAD")
    )

max_inc = df_inc.agg(F.max("total_incidents")).first()[0] or 1
df_inc_feat = df_inc.withColumn(
    "incident_proximity",
    F.round(F.col("total_incidents") / max_inc, 4)
).select("Zonal_Railway", "incident_proximity")

# Delay spike freq from existing features
df_delay_feat = spark.table("workspace.default.features_per_segment") \
    .select("Zonal_Railway", "delay_spike_freq")

# All zones scaffold
all_zones = [
    ("Central Railway",), ("Eastern Railway",), ("East Central Railway",),
    ("East Coast Railway",), ("Konkan Railway",), ("North Central Railway",),
    ("North Eastern Railway",), ("Northeast Frontier Railway",),
    ("North Western Railway",), ("Northern Railway",), ("South Central Railway",),
    ("South Eastern Railway",), ("South East Central Railway",),
    ("South Western Railway",), ("Southern Railway",), ("West Central Railway",),
    ("Western Railway",), ("Metro Railway Kolkata",), ("Production Units",),
]
df_all = spark.createDataFrame(all_zones, ["Zonal_Railway"])

# Join all features
df_features = df_all \
    .join(df_delay_feat,   on="Zonal_Railway", how="left") \
    .join(df_weather_feat, on="Zonal_Railway", how="left") \
    .join(df_inc_feat,     on="Zonal_Railway", how="left")

# Fill gaps with means
mean_delay   = df_delay_feat.agg(F.avg("delay_spike_freq")).first()[0] or 0.4
mean_weather = df_weather_feat.agg(F.avg("weather_correlation")).first()[0] or 0.5
mean_inc     = df_inc_feat.agg(F.avg("incident_proximity")).first()[0] or 0.3

df_features = df_features.fillna({
    "delay_spike_freq":    round(mean_delay, 4),
    "weather_correlation": round(mean_weather, 4),
    "incident_proximity":  round(mean_inc, 4),
})

print(f"✅ Real features table — {df_features.count()} zones")
df_features.orderBy("Zonal_Railway").show(25, truncate=False)

df_features.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.features_per_segment")

print("✅ features_per_segment saved with real data!")

In [0]:
# Join features with your real incidents data
df_incidents = spark.table("workspace.default.incidents_by_zone")

df_joined = df_features.join(df_incidents, on="Zonal_Railway", how="left")

print("Joined table:")
display(df_joined)

In [0]:
import mlflow
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression

username = spark.sql("SELECT current_user()").first()[0]
mlflow.set_experiment(f"/Users/{username}/safetrack-risk-model")

with mlflow.start_run(run_name="weighted_risk_model"):

    # Define feature columns
    feature_cols = ["delay_spike_freq", "weather_correlation", "incident_proximity"]

    # Combine features into one vector column
    assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
    df_ml = assembler.transform(df_joined)

    # Weighted scoring — simple formula
    df_scored = df_ml.withColumn(
        "risk_score",
        (F.col("delay_spike_freq")    * 0.4) +
        (F.col("weather_correlation") * 0.3) +
        (F.col("incident_proximity")  * 0.3)
    )

    # Classify into Low / Medium / High
    df_scored = df_scored.withColumn(
        "risk_label",
        F.when(F.col("risk_score") >= 0.65, "High")
         .when(F.col("risk_score") >= 0.45, "Medium")
         .otherwise("Low")
    )

    # Log parameters to MLflow
    mlflow.log_param("delay_weight",   0.4)
    mlflow.log_param("weather_weight", 0.3)
    mlflow.log_param("incident_weight",0.3)
    mlflow.log_param("high_threshold", 0.65)
    mlflow.log_param("medium_threshold", 0.45)

    # Save results
    df_scored.select(
        "Zonal_Railway", "delay_spike_freq",
        "weather_correlation", "incident_proximity",
        "risk_score", "risk_label"
    ).write.format("delta").mode("overwrite") \
     .saveAsTable("workspace.default.risk_scores")

    print("✅ Model trained and logged to MLflow!")
    display(df_scored.select("Zonal_Railway", "risk_score", "risk_label").orderBy("risk_score", ascending=False))

In [0]:
# Back-test: Do High-risk zones match zones with most real accidents?
df_risk = spark.table("workspace.default.risk_scores")
df_incidents = spark.table("workspace.default.incidents_by_zone")

# Join and compare
df_backtest = df_risk.join(df_incidents, on="Zonal_Railway", how="left")

# Get the actual accident column name
accident_col = [c for c in df_incidents.columns if "Collision" in c][0]
print("Using accident column:", accident_col)

# Flag if High-risk zones actually had accidents
df_backtest = df_backtest.withColumn(
    "had_accidents",
    F.when(F.col(accident_col) > 0, True).otherwise(False)
)

# Count how many High-risk zones had real accidents
total_high = df_backtest.filter(F.col("risk_label") == "High").count()
high_with_accidents = df_backtest.filter(
    (F.col("risk_label") == "High") & (F.col("had_accidents") == True)
).count()

if total_high > 0:
    accuracy = round((high_with_accidents / total_high) * 100, 1)
else:
    accuracy = 0

print(f"\n📊 BACK-TEST RESULTS:")
print(f"High-risk zones: {total_high}")
print(f"High-risk zones that had real accidents: {high_with_accidents}")
print(f"✅ Accuracy: {accuracy}% of High-risk zones match real accident zones")

# Log accuracy to MLflow
with mlflow.start_run(run_name="backtest_result"):
    mlflow.log_metric("backtest_accuracy_pct", accuracy)
    mlflow.log_metric("high_risk_zones", total_high)
    mlflow.log_metric("high_risk_with_accidents", high_with_accidents)
    print("✅ Accuracy logged to MLflow!")

display(df_backtest.select(
    "Zonal_Railway", "risk_label", "risk_score", accident_col, "had_accidents"
).orderBy("risk_score", ascending=False))

In [0]:
df_derail = spark.read.csv(
    "/Volumes/workspace/default/safetrack_data/RS_Session_260_AU_280_A.csv",
    header=True,
    inferSchema=True
)

# Clean column names
clean_cols = []
for col in df_derail.columns:
    new_col = col.strip().replace(" ", "_").replace("(", "").replace(")", "").replace(".", "").replace(",", "").replace("-", "_")
    clean_cols.append(new_col)

df_derail = df_derail.toDF(*clean_cols)

print("Columns:", df_derail.columns)
print("Rows:", df_derail.count())
display(df_derail)

In [0]:
# Fix zone names to match our other tables
df_derail = df_derail.withColumn(
    "Zonal_Railway",
    F.concat(F.col("Zonal_Railway"), F.lit(" Railway"))
)

# Calculate total derailments across all years per zone
df_derail = df_derail.withColumn(
    "total_derailments",
    F.col("2018_19") + F.col("2019_20") + F.col("2020_21_Covid_Year") +
    F.col("2021_22_Covid_Year") + F.col("2022_23")
)

# Save as Delta table
df_derail.write.format("delta").mode("overwrite") \
    .saveAsTable("workspace.default.derailments_by_zone")

print("✅ Derailments table created!")
display(df_derail.select("Zonal_Railway", "total_derailments").orderBy("total_derailments", ascending=False))

In [0]:
# Fix zone names to match our other tables
df_derail = df_derail.withColumn(
    "Zonal_Railway",
    F.concat(F.col("Zonal_Railway"), F.lit(" Railway"))
)

# Calculate total derailments across all years per zone
df_derail = df_derail.withColumn(
    "total_derailments",
    F.col("2018_19") + F.col("2019_20") + F.col("2020_21_Covid_Year") +
    F.col("2021_22_Covid_Year") + F.col("2022_23")
)

# Save as Delta table
df_derail.write.format("delta").mode("overwrite") \
    .saveAsTable("workspace.default.derailments_by_zone")

print("✅ Derailments table created!")
display(df_derail.select("Zonal_Railway", "total_derailments").orderBy("total_derailments", ascending=False))

In [0]:
# Load all tables
df_features = spark.table("workspace.default.features_per_segment")
df_derail = spark.table("workspace.default.derailments_by_zone")

# Normalize derailments to 0-1 score
max_derail = df_derail.agg(F.max("total_derailments")).first()[0]
df_derail = df_derail.withColumn(
    "incident_proximity",
    F.col("total_derailments") / max_derail
)

# Join features with REAL derailment data
df_joined = df_features.drop("incident_proximity").join(
    df_derail.select("Zonal_Railway", "incident_proximity", "total_derailments"),
    on="Zonal_Railway", how="left"
)

print("✅ Joined with real derailment data!")
display(df_joined)

In [0]:
# Check what zone names exist in derailments table
df_derail = spark.table("workspace.default.derailments_by_zone")
print("Derailment zones:")
df_derail.select("Zonal_Railway").show(25, truncate=False)

# Check what zone names exist in features table
df_features = spark.table("workspace.default.features_per_segment")
print("Feature zones:")
df_features.select("Zonal_Railway").show(25, truncate=False)

In [0]:
from pyspark.sql import functions as F

# Reload raw derailment CSV with correct zone names
df_derail_raw = spark.read.csv(
    "/Volumes/workspace/default/safetrack_data/RS_Session_260_AU_280_A.csv",
    header=True,
    inferSchema=True
)

# Clean column names
clean_cols = []
for col in df_derail_raw.columns:
    new_col = col.strip().replace(" ", "_").replace("(", "").replace(")", "").replace(".", "").replace(",", "").replace("-", "_")
    clean_cols.append(new_col)
df_derail_raw = df_derail_raw.toDF(*clean_cols)

# Fix zone names manually to match features table exactly
zone_mapping = {
    "Central": "Central Railway",
    "Eastern": "Eastern Railway",
    "East Central": "East Central Railway",
    "East Coast": "East Coast Railway",
    "Konkan": "Konkan Railway",
    "North Central": "North Central Railway",
    "North Eastern": "North Eastern Railway",
    "Northeast Frontier": "Northeast Frontier Railway",
    "North Western": "North Western Railway",
    "Northern": "Northern Railway",
    "South Central": "South Central Railway",
    "South Eastern": "South Eastern Railway",
    "South East Central": "South East Central Railway",
    "South Western": "South Western Railway",
    "Southern": "Southern Railway",
    "West Central": "West Central Railway",
    "Western": "Western Railway",
    "Metro": "Metro Railway Kolkata",
    "Total": "Production Units"
}

# Apply mapping
from pyspark.sql.functions import create_map, lit
from itertools import chain

mapping_expr = create_map([lit(x) for x in chain(*zone_mapping.items())])
df_derail_fixed = df_derail_raw.withColumn(
    "Zonal_Railway",
    mapping_expr[F.col("Zonal_Railway")]
)

# Calculate total derailments
df_derail_fixed = df_derail_fixed.withColumn(
    "total_derailments",
    F.col("2018_19") + F.col("2019_20") + F.col("2020_21_Covid_Year") +
    F.col("2021_22_Covid_Year") + F.col("2022_23")
)

# Save fixed table
df_derail_fixed.write.format("delta").mode("overwrite") \
    .saveAsTable("workspace.default.derailments_by_zone")

print("✅ Fixed zone names!")
display(df_derail_fixed.select("Zonal_Railway", "total_derailments").orderBy("total_derailments", ascending=False))

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType
import math

# Step 1 — Pull coordinates from both tables into Python
weather_coords = spark.table("weather_history") \
    .select("station_name", "latitude", "longitude") \
    .distinct() \
    .collect()

station_coords = spark.table("stations_cleaned") \
    .select("station_code", "station_name", "zone", "latitude", "longitude") \
    .filter(F.col("latitude").isNotNull()) \
    .filter(F.col("longitude").isNotNull()) \
    .collect()

# Step 2 — For each weather station, find the closest station by distance
def haversine(lat1, lon1, lat2, lon2):
    """Straight-line distance in km between two coordinates"""
    R = 6371
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = math.sin(dlat/2)**2 + math.cos(lat1)*math.cos(lat2)*math.sin(dlon/2)**2
    return R * 2 * math.asin(math.sqrt(a))

# Build mapping: weather station_name → station_code + zone
weather_to_code = {}
for w in weather_coords:
    best_match = None
    best_dist = float("inf")
    fo

In [0]:
# Load tables
df_features = spark.table("workspace.default.features_per_segment")
df_derail = spark.table("workspace.default.derailments_by_zone")

# Remove the Total row
df_derail = df_derail.filter(F.col("Zonal_Railway") != "Production Units")

# Normalize derailments to 0-1
max_derail = df_derail.agg(F.max("total_derailments")).first()[0]
df_derail = df_derail.withColumn(
    "incident_proximity",
    F.col("total_derailments") / max_derail
)

# Join with features
df_joined = df_features.drop("incident_proximity").join(
    df_derail.select("Zonal_Railway", "incident_proximity", "total_derailments"),
    on="Zonal_Railway", how="left"
)

print("✅ Joined successfully!")
display(df_joined.select("Zonal_Railway", "delay_spike_freq", "weather_correlation", "incident_proximity", "total_derailments"))

In [0]:
username = spark.sql("SELECT current_user()").first()[0]
mlflow.set_experiment(f"/Users/{username}/safetrack-risk-model")

with mlflow.start_run(run_name="real_derailment_final_model"):

    df_scored = df_joined.withColumn(
        "risk_score",
        (F.col("delay_spike_freq")    * 0.4) +
        (F.col("weather_correlation") * 0.3) +
        (F.col("incident_proximity")  * 0.3)
    ).withColumn(
        "risk_label",
        F.when(F.col("risk_score") >= 0.65, "High")
         .when(F.col("risk_score") >= 0.45, "Medium")
         .otherwise("Low")
    )

    # Back-test
    total_high = df_scored.filter(F.col("risk_label") == "High").count()
    high_with_derails = df_scored.filter(
        (F.col("risk_label") == "High") & (F.col("total_derailments") > 5)
    ).count()
    accuracy = round((high_with_derails / total_high) * 100, 1) if total_high > 0 else 0

    # Log to MLflow
    mlflow.log_param("delay_weight", 0.4)
    mlflow.log_param("weather_weight", 0.3)
    mlflow.log_param("incident_weight", 0.3)
    mlflow.log_param("high_threshold", 0.65)
    mlflow.log_metric("backtest_accuracy_pct", accuracy)
    mlflow.log_metric("high_risk_zones", total_high)
    mlflow.log_metric("high_risk_with_derailments", high_with_derails)

    # Save final risk scores
    df_scored.select(
        "Zonal_Railway", "delay_spike_freq", "weather_correlation",
        "incident_proximity", "total_derailments", "risk_score", "risk_label"
    ).write.format("delta").mode("overwrite") \
     .saveAsTable("workspace.default.risk_scores")

    print(f"✅ Final model trained!")
    print(f"📊 Back-test accuracy: {accuracy}%")
    print(f"High-risk zones: {total_high}")
    print(f"High-risk zones with real derailments > 5: {high_with_derails}")

    display(df_scored.select(
        "Zonal_Railway", "risk_label", "risk_score", "total_derailments"
    ).orderBy("risk_score", ascending=False))

In [0]:
username = spark.sql("SELECT current_user()").first()[0]
mlflow.set_experiment(f"/Users/{username}/safetrack-risk-model")

with mlflow.start_run(run_name="real_derailment_final_model"):

    df_scored = df_joined.withColumn(
        "risk_score",
        (F.col("delay_spike_freq")    * 0.4) +
        (F.col("weather_correlation") * 0.3) +
        (F.col("incident_proximity")  * 0.3)
    ).withColumn(
        "risk_label",
        F.when(F.col("risk_score") >= 0.65, "High")
         .when(F.col("risk_score") >= 0.45, "Medium")
         .otherwise("Low")
    )

    # Back-test
    total_high = df_scored.filter(F.col("risk_label") == "High").count()
    high_with_derails = df_scored.filter(
        (F.col("risk_label") == "High") & (F.col("total_derailments") > 5)
    ).count()
    accuracy = round((high_with_derails / total_high) * 100, 1) if total_high > 0 else 0

    # Log to MLflow
    mlflow.log_param("delay_weight", 0.4)
    mlflow.log_param("weather_weight", 0.3)
    mlflow.log_param("incident_weight", 0.3)
    mlflow.log_metric("backtest_accuracy_pct", accuracy)
    mlflow.log_metric("high_risk_zones", total_high)
    mlflow.log_metric("high_risk_with_derailments", high_with_derails)

    # ✅ Added overwriteSchema to fix the merge error
    df_scored.select(
        "Zonal_Railway", "delay_spike_freq", "weather_correlation",
        "incident_proximity", "total_derailments", "risk_score", "risk_label"
    ).write.format("delta").mode("overwrite") \
     .option("overwriteSchema", "true") \
     .saveAsTable("workspace.default.risk_scores")

    print(f"✅ Final model trained!")
    print(f"📊 Back-test accuracy: {accuracy}%")
    print(f"High-risk zones: {total_high}")
    print(f"High-risk zones with real derailments > 5: {high_with_derails}")

    display(df_scored.select(
        "Zonal_Railway", "risk_label", "risk_score", "total_derailments"
    ).orderBy("risk_score", ascending=False))

In [0]:
# Print your MLflow experiment URL
username = spark.sql("SELECT current_user()").first()[0]
print(f"Go to this experiment in Databricks:")
print(f"Experiments → /Users/{username}/safetrack-risk-model")
print(f"Take a screenshot showing all runs, params, and metrics")

In [0]:
# ============================================
# SAFETRACK — BACK-TEST ACCURACY SUMMARY
# ============================================

print("=" * 55)
print("   SAFETRACK RISK MODEL — BACK-TEST RESULTS")
print("=" * 55)
print()
print("Model: Weighted scoring (Delay 40% + Weather 30% + Incident 30%)")
print()
print("Back-test method:")
print("  → Compared model's High-risk zones against")
print("    real historical derailment data (2018–2023)")
print()
print("Results:")
print("  • Total High-risk zones identified : 2")
print("  • High-risk zones with real derailments > 5 : 2")
print()
print("  ✅ BACK-TEST ACCURACY: 100%")
print()
print("Key findings:")
print("  🔴 Northern Railway — Score: 0.758 — 25 real derailments")
print("  🔴 Central Railway  — Score: 0.747 — 22 real derailments")
print("=" * 55)